In [1]:
import google.generativeai as genai
import os
import re

# --- Configuration ---
API_KEY_FILE = "apikey.txt"
MODEL_NAME = 'gemini-2.5-flash-preview-05-20' # 最新のモデルを指定、または 'gemini-pro' など
# MODEL_NAME = 'gemini-pro'

def get_api_key(filepath=API_KEY_FILE):
    """指定されたファイルからAPIキーを読み込む"""
    try:
        with open(filepath, "r") as f:
            return f.read().strip()
    except FileNotFoundError:
        print(f"エラー: APIキーファイル '{filepath}' が見つかりません。")
        return None

def generate_senryu_from_llm(api_key, theme, num_senryu=10, context_hint=""):
    """
    LLMを使用して指定されたお題で川柳を生成する。
    """
    if not api_key:
        print("APIキーが設定されていません。")
        return None

    genai.configure(api_key=api_key)
    try:
        model = genai.GenerativeModel(MODEL_NAME)
    except Exception as e:
        print(f"モデルの初期化中にエラーが発生しました ({MODEL_NAME}): {e}")
        return None

    # プロンプトの作成
    prompt = f"""お題は「{theme}」です。
以下の指示に従って、川柳を{num_senryu}個作成してください。

指示:
1. 川柳は五七五の形式でお願いします。
2. ユーモラスなもの、風刺的なもの、日常の一コマ、情景が浮かぶものなど、自由に発想してください。
3. 「{theme}」という言葉を直接使っても使わなくても構いませんが、お題が想起される内容にしてください。
{context_hint}

それでは、川柳を{num_senryu}個、それぞれ改行して提示してください。番号は不要です。
"""

    print("--- LLMに送信するプロンプトの概要 ---")
    print(f"お題: {theme}")
    print(f"個数: {num_senryu}")
    if context_hint:
        print(f"コンテキストヒントあり")
    print("------------------------------------")

    try:
        print("\nLLMに川柳生成をリクエスト中...")
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"LLMからの応答生成中にエラーが発生しました: {e}")
        return None

def display_senryu(senryu_text, num_expected):
    """
    LLMから返されたテキストを整形して川柳を表示する。
    """
    if not senryu_text:
        print("表示する川柳がありません。")
        return

    print("\n--- 生成された川柳 ---")
    # 改行で分割し、空行や余分なスペースを除去
    lines = [line.strip() for line in senryu_text.strip().split('\n') if line.strip()]

    count = 0
    for i, line in enumerate(lines):
        # 簡単なフィルタリング（例：明らかに川柳でない短い行や長すぎる行を除外する試み）
        # ただし、LLMの出力は多様なので完璧なフィルタは難しい
        # ここでは、句読点や空白を除いた文字数が10～25字程度のものを川柳候補とする
        # (五七五は17字だが、字余り・字足らずや空白の扱いを考慮)
        text_for_len_check = re.sub(r'[、。！？「」（）『』]', '', line).replace(" ", "").replace("　", "")
        if 10 <= len(text_for_len_check) <= 25: # この範囲は調整の余地あり
            count += 1
            print(f"{count}. {line}")
            if count >= num_expected:
                break
        elif len(lines) <= num_expected * 1.5 : # フィルタにかからなくても、行数が少なければ表示試行
            count += 1
            print(f"{count}. {line} (形式注意)") # 形式が異なる可能性があることを示す
            if count >= num_expected:
                break


    if count == 0:
        print("川柳形式のテキストを抽出できませんでした。生の応答を以下に示します：")
        print(senryu_text)
    elif count < num_expected:
        print(f"\n注意: {count}個の川柳（またはそれに近い行）が抽出されました。期待した数 ({num_expected}個) と異なる場合があります。")
        if count < len(lines) and count < num_expected : # まだ表示していない行があれば追加で表示試行
             print("\n--- その他LLM応答行 (参考) ---")
             for extra_line in lines[count:]:
                 print(extra_line)


if __name__ == "__main__":
    api_key = get_api_key()

    if api_key:
        senryu_theme = "国立"
        number_of_senryu = 10

        # 現在の場所に関するコンテキスト情報をヒントとして追加
        # この情報は model.generate_content() の前に設定されるグローバルなものではないため、
        # プロンプトに明示的に含める。
        location_context_hint = f"""
ヒント:
「国立」には、国の施設や機関（例：国立競技場、国立大学、国立博物館、国立公園など）を指す「こくりつ」と、
地名である東京都「国立市（くにたちし）」（学園都市、大学通り、桜並木などが知られる）を指す「くにたち」の二つの意味合いがあります。
これらの要素を自由に発想に含めてください。
"""
        print(f"お題「{senryu_theme}」で川柳を{number_of_senryu}個生成します。")
        generated_text = generate_senryu_from_llm(api_key, senryu_theme, number_of_senryu, location_context_hint)

        if generated_text:
            display_senryu(generated_text, number_of_senryu)
        else:
            print("川柳の生成に失敗しました。")
    else:
        print("APIキーがないため、プログラムを実行できません。")

c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


お題「国立」で川柳を10個生成します。
--- LLMに送信するプロンプトの概要 ---
お題: 国立
個数: 10
コンテキストヒントあり
------------------------------------

LLMに川柳生成をリクエスト中...

--- 生成された川柳 ---
1. 赤本を 抱いて挑むは 国立大
2. 国立の 桜トンネル 春を呼ぶ
3. 歓声が 響く国立 夢舞台
4. 学園の 香り漂う 国立路
5. 太古から 国立展示 知の宝
6. 国の施設 予算は常に 大規模だ
7. 大学の 通り散歩で 昼下がり
8. 広大に 緑あふれる 国立園
9. 銀杏散る 黄色い絨毯 くにたちの
10. 安い学費 親もニッコリ 国立だ
